# Gemma 4 Developer Agent — Kaggle submission notebook

Runs the offline agent harness end to end:

1. Locate the attached bundle dataset (code + wheels + weights + tasks).
2. Install the bundled wheels with `--no-index` (internet stays OFF).
3. Start the local vLLM server on the bundled Gemma 4 weights.
4. Run the agent over every task; each task gets a disposable repo copy.
5. Write `/kaggle/working/predictions.jsonl` (one JSON object per task:
   `instance_id`, `model_patch`, `model_name_or_path`).

Total wall clock is capped by `configs/default_agent.yaml`
(`run.global_timeout_seconds`, default 11.5 h of the 12 h limit).


In [ ]:
import json, os, sys
from pathlib import Path

# --- 1. Locate the bundle -------------------------------------------------
INPUT = Path("/kaggle/input")
bundle = None
for manifest in INPUT.rglob("manifest.json"):
    bundle = manifest.parent
    break
assert bundle is not None, "bundle dataset not attached (manifest.json not found)"
print("bundle:", bundle)

CODE = bundle / "code"
sys.path.insert(0, str(CODE / "src"))
os.environ["GEMMA4_MODEL_PATH"] = str(bundle / "models" / "gemma-4")
WORK = Path("/kaggle/working")
print("model:", os.environ["GEMMA4_MODEL_PATH"])


In [ ]:
# --- 2. Offline install of bundled wheels ---------------------------------
import subprocess, sys
wheels = bundle / "wheels"
if any(wheels.glob("*.whl")):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index",
         "--find-links", str(wheels), "pyyaml"],
        check=True,
    )
else:
    print("no wheels bundled; assuming pyyaml is preinstalled")
import yaml  # noqa: E402  (verifies the offline install worked)
print("pyyaml OK")


In [ ]:
# --- 3. Start vLLM on the bundled weights ---------------------------------
from engine.vllm_server import VLLMServer, load_vllm_config
from engine.client import LocalVLLMClient

vllm_cfg = load_vllm_config(CODE / "configs" / "vllm_config.yaml")
server = VLLMServer(vllm_cfg)
base_url = server.start()
print("vLLM healthy at", base_url)
client = LocalVLLMClient(base_url=base_url, model=vllm_cfg["model"]["served_name"])


In [ ]:
# --- 4. Run the agent over all tasks ---------------------------------------
from agent.core import AgentConfig
from pipeline.runner import TaskSpec, run_tasks

tasks_file = bundle / "tasks" / "tasks.json"
tasks = TaskSpec.load_many(tasks_file)
print(f"{len(tasks)} tasks loaded")

agent_cfg = AgentConfig.from_yaml(CODE / "configs" / "default_agent.yaml")
try:
    summary = run_tasks(
        tasks,
        client_factory=lambda task: client,
        agent_config=agent_cfg,
        config_path=CODE / "configs" / "default_agent.yaml",
        output_dir=WORK / "run",
    )
    print("counts:", summary.counts)
finally:
    server.stop()
    print("vLLM stopped")


In [ ]:
# --- 5. Write predictions ---------------------------------------------------
import json
from pathlib import Path

results = json.loads((WORK / "run" / "results.json").read_text())
patch_by_task = {}
for outcome in results["outcomes"]:
    if outcome["patch_file"]:
        patch_by_task[outcome["task_id"]] = Path(outcome["patch_file"]).read_text()

predictions = WORK / "predictions.jsonl"
with open(predictions, "w") as fh:
    for outcome in results["outcomes"]:
        fh.write(json.dumps({
            "instance_id": outcome["task_id"],
            "model_patch": patch_by_task.get(outcome["task_id"], ""),
            "model_name_or_path": "gemma-4",
        }) + "\n")
print("wrote", predictions, "with", len(results["outcomes"]), "predictions")
